In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q kaggle scikit-learn

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
BACKUP_DIR = Path("/content/drive/MyDrive/PlantLeafNet")

for folder in ["splits", "models", "results"]:
    path = BACKUP_DIR / folder
    print(folder, ":", path.exists())

splits : True
models : True
results : True


In [8]:
import shutil
from pathlib import Path

BACKUP_DIR = Path("/content/drive/MyDrive/PlantLeafNet")
PROJECT_DIR = Path("/content/PlantLeafNet")

SPLIT_DIR = PROJECT_DIR / "splits"
MODEL_DIR = PROJECT_DIR / "models"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Restore split CSVs
for filename in ["train.csv", "validation.csv", "test.csv"]:
    shutil.copy2(
        BACKUP_DIR / "splits" / filename,
        SPLIT_DIR / filename
    )

# Restore CNN checkpoints
for filename in ["best_cnn_baseline.pth", "best_weighted_cnn.pth"]:
    shutil.copy2(
        BACKUP_DIR / "models" / filename,
        MODEL_DIR / filename
    )

print("Restoration completed!")

Restoration completed!


In [9]:
import shutil
from pathlib import Path

BACKUP_DIR = Path("/content/drive/MyDrive/PlantLeafNet")
PROJECT_DIR = Path("/content/PlantLeafNet")

SPLIT_DIR = PROJECT_DIR / "splits"
MODEL_DIR = PROJECT_DIR / "models"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Restore split CSVs
for filename in ["train.csv", "validation.csv", "test.csv"]:
    shutil.copy2(
        BACKUP_DIR / "splits" / filename,
        SPLIT_DIR / filename
    )

# Restore CNN checkpoints
for filename in ["best_cnn_baseline.pth", "best_weighted_cnn.pth"]:
    shutil.copy2(
        BACKUP_DIR / "models" / filename,
        MODEL_DIR / filename
    )

print("Restoration completed!")

Restoration completed!


In [10]:
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for filename in [
    "cnn_baseline_history.csv",
    "weighted_cnn_history.csv",
    "cnn_comparison.csv"
]:
    source = BACKUP_DIR / "results" / filename

    if source.exists():
        shutil.copy2(source, RESULTS_DIR / filename)
        print("Restored:", filename)

Restored: cnn_baseline_history.csv
Restored: weighted_cnn_history.csv
Restored: cnn_comparison.csv


In [11]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

print("\nColumns:", train_df.columns.tolist())

Training: 37998
Validation: 8142
Testing: 8143

Columns: ['Image_Path', 'Class']


In [12]:
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/PlantLeafNet

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:23<00:00, 91.6MB/s]



In [13]:
import zipfile

zip_files = list(PROJECT_DIR.glob("*.zip"))

with zipfile.ZipFile(zip_files[0], "r") as zip_ref:
    zip_ref.extractall(PROJECT_DIR)

print("Dataset extracted!")

Dataset extracted!


In [14]:
COLOR_DIR = Path(
    "/content/PlantLeafNet/plantvillage dataset/color"
)

print("Color directory exists:", COLOR_DIR.exists())
print("Number of classes:", len([
    p for p in COLOR_DIR.iterdir() if p.is_dir()
]))

Color directory exists: True
Number of classes: 38


In [15]:
for df in [train_df, val_df, test_df]:
    df["Image_Path"] = df["Image_Path"].apply(
        lambda old_path: str(
            COLOR_DIR / Path(old_path).parent.name / Path(old_path).name
        )
    )

In [16]:
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing = sum(
        not Path(path).exists()
        for path in df["Image_Path"]
    )

    print(f"{name}: {missing} missing images out of {len(df)}")

Train: 0 missing images out of 37998
Validation: 0 missing images out of 8142
Test: 0 missing images out of 8143


In [17]:
classes = sorted(train_df["Class"].unique())

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(classes)
}

print("Number of classes:", len(classes))
print("First five classes:", classes[:5])

Number of classes: 38
First five classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [18]:
weights = models.MobileNet_V3_Large_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = weights.transforms()
test_transform = weights.transforms()

In [19]:
class PlantDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(row["Image_Path"]).convert("RGB")

        label = class_to_idx[row["Class"]]

        if self.transform:
            image = self.transform(image)

        return image, label

In [20]:
train_dataset = PlantDataset(train_df, train_transform)
val_dataset = PlantDataset(val_df, val_transform)
test_dataset = PlantDataset(test_df, test_transform)

In [21]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [22]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Image range:", images.min().item(), images.max().item())

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Image range: -2.1179039478302 2.640000104904175


In [23]:
weights = models.MobileNet_V3_Large_Weights.DEFAULT

model = models.mobilenet_v3_large(weights=weights)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 55.5MB/s]


Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=1000, bias=True)
)


In [24]:
num_features = model.classifier[3].in_features

model.classifier[3] = nn.Linear(
    num_features,
    len(classes)
)

model = model.to(device)

print(model.classifier)

Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=38, bias=True)
)


In [25]:
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

In [26]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.2f}%"
)

Total parameters: 4250710
Trainable parameters: 1278758
Trainable percentage: 30.08%


In [27]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

print("Training configuration ready!")

Training configuration ready!


In [28]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

    return running_loss / len(loader), 100 * correct / total


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    return running_loss / len(loader), 100 * correct / total

In [29]:
EPOCHS = 5

history_mobilenet = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_accuracy = 0.0

MODEL_PATH = "/content/PlantLeafNet/best_mobilenet_v3.pth"

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = validate_one_epoch(
        model, val_loader, criterion, device
    )

    scheduler.step(val_loss)

    history_mobilenet["train_loss"].append(train_loss)
    history_mobilenet["train_accuracy"].append(train_acc)
    history_mobilenet["val_loss"].append(val_loss)
    history_mobilenet["val_accuracy"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print("Best MobileNet model saved!")

print("\nBest validation accuracy:", best_val_accuracy)


Epoch 1/5
Train Loss: 1.0433
Train Accuracy: 91.16%
Validation Loss: 0.9111
Validation Accuracy: 94.76%
Best MobileNet model saved!

Epoch 2/5
Train Loss: 0.8928
Train Accuracy: 95.79%
Validation Loss: 0.8552
Validation Accuracy: 96.86%
Best MobileNet model saved!

Epoch 3/5
Train Loss: 0.8674
Train Accuracy: 96.56%
Validation Loss: 0.8593
Validation Accuracy: 96.50%

Epoch 4/5
Train Loss: 0.8525
Train Accuracy: 97.21%
Validation Loss: 0.8532
Validation Accuracy: 96.60%

Epoch 5/5
Train Loss: 0.8429
Train Accuracy: 97.38%
Validation Loss: 0.8337
Validation Accuracy: 97.33%
Best MobileNet model saved!

Best validation accuracy: 97.33480717268485


In [30]:
import shutil
import os

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PlantLeafNet/models"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/PlantLeafNet/results"

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

shutil.copy2(
    MODEL_PATH,
    f"{DRIVE_MODEL_DIR}/best_mobilenet_v3.pth"
)

pd.DataFrame(history_mobilenet).to_csv(
    f"{DRIVE_RESULTS_DIR}/mobilenet_training_history.csv",
    index_label="Epoch"
)

print("MobileNet checkpoint and history backed up!")

MobileNet checkpoint and history backed up!


In [32]:
from sklearn.metrics import classification_report

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint)
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())

report = classification_report(
    all_labels,
    all_predictions,
    labels=range(len(classes)),
    target_names=classes,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print(report_df.loc[["macro avg", "weighted avg"]].round(4))

              precision  recall  f1-score  support
macro avg        0.9670  0.9648    0.9655   8142.0
weighted avg     0.9739  0.9733    0.9733   8142.0


# **selective fine-tuning**

In [33]:
for i, layer in enumerate(model.features):
    print(i, layer)

0 Conv2dNormActivation(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (2): Hardswish()
)
1 InvertedResidual(
  (block): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (1): Conv2dNormActivation(
      (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
    )
  )
)
2 InvertedResidual(
  (block): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    

In [34]:
# First freeze the entire feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze the final three feature blocks
for layer in model.features[14:]:
    for param in layer.parameters():
        param.requires_grad = True

# Keep the classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# Count trainable parameters
trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

total = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", trainable)
print("Total parameters:", total)
print(f"Trainable percentage: {100 * trainable / total:.2f}%")

Trainable parameters: 3028998
Total parameters: 4250710
Trainable percentage: 71.26%


In [35]:
optimizer = optim.Adam([
    {
        "params": model.features[14:].parameters(),
        "lr": 1e-5
    },
    {
        "params": model.classifier.parameters(),
        "lr": 1e-4
    }
])

In [36]:
FINE_TUNE_EPOCHS = 3

fine_tune_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

# Start from the best feature-extraction checkpoint
model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device, weights_only=True)
)

# Reapply the intended trainable layers after loading
for param in model.features.parameters():
    param.requires_grad = False

for layer in model.features[14:]:
    for param in layer.parameters():
        param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

# Recreate optimizer after loading
optimizer = optim.Adam([
    {"params": model.features[14:].parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4}
])

best_finetuned_accuracy = 0.0

FINETUNED_PATH = "/content/PlantLeafNet/best_mobilenet_finetuned.pth"

for epoch in range(FINE_TUNE_EPOCHS):

    print(f"\nFine-tuning Epoch {epoch + 1}/{FINE_TUNE_EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = validate_one_epoch(
        model, val_loader, criterion, device
    )

    fine_tune_history["train_loss"].append(train_loss)
    fine_tune_history["train_accuracy"].append(train_acc)
    fine_tune_history["val_loss"].append(val_loss)
    fine_tune_history["val_accuracy"].append(val_acc)

    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")

    if val_acc > best_finetuned_accuracy:
        best_finetuned_accuracy = val_acc
        torch.save(model.state_dict(), FINETUNED_PATH)
        print("Best fine-tuned model saved!")

print("\nBest fine-tuned accuracy:", best_finetuned_accuracy)


Fine-tuning Epoch 1/3
Train Accuracy: 98.79%
Validation Accuracy: 98.61%
Validation Loss: 0.7834
Best fine-tuned model saved!

Fine-tuning Epoch 2/3
Train Accuracy: 99.15%
Validation Accuracy: 98.78%
Validation Loss: 0.7736
Best fine-tuned model saved!

Fine-tuning Epoch 3/3
Train Accuracy: 99.30%
Validation Accuracy: 98.92%
Validation Loss: 0.7649
Best fine-tuned model saved!

Best fine-tuned accuracy: 98.91918447555884


In [37]:
import shutil
import os
import pandas as pd

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PlantLeafNet/models"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/PlantLeafNet/results"

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

shutil.copy2(
    FINETUNED_PATH,
    f"{DRIVE_MODEL_DIR}/best_mobilenet_finetuned.pth"
)

pd.DataFrame(fine_tune_history).to_csv(
    f"{DRIVE_RESULTS_DIR}/mobilenet_finetuning_history.csv",
    index_label="Epoch"
)

print("Fine-tuned model and history backed up!")

Fine-tuned model and history backed up!


In [38]:
model.load_state_dict(
    torch.load(FINETUNED_PATH, map_location=device, weights_only=True)
)

model.eval()

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [39]:
from sklearn.metrics import classification_report, accuracy_score

test_labels = []
test_predictions = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        test_labels.extend(labels.numpy())
        test_predictions.extend(predictions.cpu().numpy())

print(
    "Test Accuracy:",
    f"{accuracy_score(test_labels, test_predictions) * 100:.2f}%"
)

Test Accuracy: 98.89%


In [40]:
test_report = classification_report(
    test_labels,
    test_predictions,
    labels=range(len(classes)),
    target_names=classes,
    output_dict=True,
    zero_division=0
)

test_report_df = pd.DataFrame(test_report).transpose()

test_report_df.loc[
    ["macro avg", "weighted avg"]
].round(4)

,precision,recall,f1-score,support
macro avg,0.985,0.9829,0.9839,8143.0
weighted avg,0.989,0.9889,0.9889,8143.0


In [41]:
test_report_df.to_csv(
    f"{DRIVE_RESULTS_DIR}/mobilenet_test_classification_report.csv"
)

In [42]:
import json
from sklearn.metrics import confusion_matrix

# Save class mapping
with open("/content/PlantLeafNet/class_mapping.json", "w") as f:
    json.dump(class_to_idx, f, indent=4)

# Save test confusion matrix
cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=range(len(classes))
)

pd.DataFrame(
    cm,
    index=classes,
    columns=classes
).to_csv("/content/PlantLeafNet/mobilenet_test_confusion_matrix.csv")

# Back up to Drive
shutil.copy2(
    "/content/PlantLeafNet/class_mapping.json",
    "/content/drive/MyDrive/PlantLeafNet/class_mapping.json"
)

shutil.copy2(
    "/content/PlantLeafNet/mobilenet_test_confusion_matrix.csv",
    "/content/drive/MyDrive/PlantLeafNet/results/mobilenet_test_confusion_matrix.csv"
)

print("MobileNet results secured!")

MobileNet results secured!


# ***EfficientNetV2-S***

In [51]:
weights_eff = models.EfficientNet_V2_S_Weights.DEFAULT

efficientnet = models.efficientnet_v2_s(
    weights=weights_eff
)

print(efficientnet.classifier)

In [44]:
num_features = efficientnet.classifier[1].in_features

efficientnet.classifier[1] = nn.Linear(
    num_features,
    len(classes)
)

efficientnet = efficientnet.to(device)

print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=38, bias=True)
)


In [49]:
# Official EfficientNetV2-S preprocessing
val_transform_eff = weights_eff.transforms()
test_transform_eff = weights_eff.transforms()

# Training augmentation
train_transform_eff = transforms.Compose([
    transforms.RandomResizedCrop(384),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=val_transform_eff.mean,
        std=val_transform_eff.std
    )
])

print("Transformations ready!")

Transformations ready!


In [53]:
train_dataset_eff = PlantDataset(
    train_df, train_transform_eff
)

val_dataset_eff = PlantDataset(
    val_df, val_transform_eff
)

test_dataset_eff = PlantDataset(
    test_df, test_transform_eff
)

train_loader_eff = DataLoader(
    train_dataset_eff,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_eff = DataLoader(
    val_dataset_eff,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_eff = DataLoader(
    test_dataset_eff,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [54]:
for param in efficientnet.features.parameters():
    param.requires_grad = False

for param in efficientnet.classifier.parameters():
    param.requires_grad = True

total_params = sum(
    p.numel() for p in efficientnet.parameters()
)

trainable_params = sum(
    p.numel() for p in efficientnet.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.2f}%"
)

Total parameters: 20226166
Trainable parameters: 48678
Trainable percentage: 0.24%


In [55]:
criterion_eff = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer_eff = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        efficientnet.parameters()
    ),
    lr=0.001
)

print("EfficientNetV2-S ready for training!")

EfficientNetV2-S ready for training!


In [57]:
criterion_eff = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer_eff = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)

print("Loss and optimizer configured!")

Loss and optimizer configured!


In [58]:
EPOCHS_EFF = 5

history_eff = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc_eff = 0.0

for epoch in range(EPOCHS_EFF):

    train_loss, train_acc = train_one_epoch(
        efficientnet, train_loader_eff,
        criterion_eff, optimizer_eff, device
    )

    val_loss, val_acc = validate_one_epoch(
        efficientnet, val_loader_eff,
        criterion_eff, device
    )

    history_eff["train_loss"].append(train_loss)
    history_eff["train_acc"].append(train_acc)
    history_eff["val_loss"].append(val_loss)
    history_eff["val_acc"].append(val_acc)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_EFF}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

    if val_acc > best_val_acc_eff:
        best_val_acc_eff = val_acc

        torch.save(
            efficientnet.state_dict(),
            "/content/PlantLeafNet/best_efficientnet_v2_s.pth"
        )

        print("Best model checkpoint saved!")

print("\nBest Validation Accuracy:", best_val_acc_eff)


Epoch [1/5]
Train Loss: 1.7341 | Train Acc: 67.99%
Val Loss:   1.2905 | Val Acc:   84.99%
Best model checkpoint saved!

Epoch [2/5]
Train Loss: 1.4420 | Train Acc: 77.19%
Val Loss:   1.1971 | Val Acc:   87.47%
Best model checkpoint saved!

Epoch [3/5]
Train Loss: 1.4103 | Train Acc: 78.27%
Val Loss:   1.1858 | Val Acc:   87.57%
Best model checkpoint saved!

Epoch [4/5]
Train Loss: 1.3986 | Train Acc: 78.65%
Val Loss:   1.1451 | Val Acc:   88.92%
Best model checkpoint saved!

Epoch [5/5]
Train Loss: 1.3995 | Train Acc: 78.91%
Val Loss:   1.1451 | Val Acc:   89.04%
Best model checkpoint saved!

Best Validation Accuracy: 89.04446082043724


In [59]:
import os
import shutil
import pandas as pd

MODEL_DIR = "/content/drive/MyDrive/PlantLeafNet/models"
RESULT_DIR = "/content/drive/MyDrive/PlantLeafNet/results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

shutil.copy(
    "/content/PlantLeafNet/best_efficientnet_v2_s.pth",
    MODEL_DIR
)

pd.DataFrame(history_eff).to_csv(
    f"{RESULT_DIR}/efficientnet_v2_s_history.csv",
    index=False
)

print("EfficientNet model and training history backed up!")

EfficientNet model and training history backed up!


In [60]:
for i, layer in enumerate(efficientnet.features):
    print(i, layer.__class__.__name__)

0 Conv2dNormActivation
1 Sequential
2 Sequential
3 Sequential
4 Sequential
5 Sequential
6 Sequential
7 Conv2dNormActivation


In [61]:
# Freeze the entire feature extractor
for param in efficientnet.features.parameters():
    param.requires_grad = False

# Unfreeze the final two feature blocks
for layer in efficientnet.features[6:]:
    for param in layer.parameters():
        param.requires_grad = True

# Keep the classifier trainable
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in efficientnet.parameters())

trainable_params = sum(
    p.numel() for p in efficientnet.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable percentage:", round(trainable_params / total_params * 100, 2), "%")

Total parameters: 20226166
Trainable parameters: 14940750
Trainable percentage: 73.87 %


In [62]:
optimizer_eff = optim.Adam([
    {
        "params": efficientnet.features[6:].parameters(),
        "lr": 1e-5
    },
    {
        "params": efficientnet.classifier.parameters(),
        "lr": 1e-4
    }
])

In [63]:
checkpoint = torch.load(
    "/content/PlantLeafNet/best_efficientnet_v2_s.pth",
    map_location=device,
    weights_only=True
)

efficientnet.load_state_dict(checkpoint)

<All keys matched successfully>

In [64]:
EPOCHS_FINE_EFF = 3

history_fine_eff = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc_fine_eff = 0.0

for epoch in range(EPOCHS_FINE_EFF):

    train_loss, train_acc = train_one_epoch(
        efficientnet,
        train_loader_eff,
        criterion_eff,
        optimizer_eff,
        device
    )

    val_loss, val_acc = validate_one_epoch(
        efficientnet,
        val_loader_eff,
        criterion_eff,
        device
    )

    history_fine_eff["train_loss"].append(train_loss)
    history_fine_eff["train_acc"].append(train_acc)
    history_fine_eff["val_loss"].append(val_loss)
    history_fine_eff["val_acc"].append(val_acc)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FINE_EFF}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc_fine_eff:
        best_val_acc_fine_eff = val_acc

        torch.save(
            efficientnet.state_dict(),
            "/content/PlantLeafNet/best_efficientnet_finetuned.pth"
        )

        print("Best fine-tuned checkpoint saved!")

print("\nBest Fine-tuned Validation Accuracy:", best_val_acc_fine_eff)


Epoch [1/3]
Train Loss: 1.1631 | Train Acc: 87.35%
Val Loss: 0.8867 | Val Acc: 97.00%
Best fine-tuned checkpoint saved!

Epoch [2/3]
Train Loss: 1.0266 | Train Acc: 91.99%
Val Loss: 0.8429 | Val Acc: 97.67%
Best fine-tuned checkpoint saved!

Epoch [3/3]
Train Loss: 0.9665 | Train Acc: 93.76%
Val Loss: 0.8114 | Val Acc: 98.53%
Best fine-tuned checkpoint saved!

Best Fine-tuned Validation Accuracy: 98.52616064848931


In [65]:
import os
import shutil
import pandas as pd

MODEL_DIR = "/content/drive/MyDrive/PlantLeafNet/models"
RESULT_DIR = "/content/drive/MyDrive/PlantLeafNet/results"

shutil.copy(
    "/content/PlantLeafNet/best_efficientnet_finetuned.pth",
    MODEL_DIR
)

pd.DataFrame(history_fine_eff).to_csv(
    f"{RESULT_DIR}/efficientnet_finetuning_history.csv",
    index=False
)

print("Fine-tuned EfficientNet backed up!")

Fine-tuned EfficientNet backed up!


In [67]:
efficientnet.load_state_dict(
    torch.load(
        "/content/PlantLeafNet/best_efficientnet_finetuned.pth",
        map_location=device,
        weights_only=True
    )
)

efficientnet.eval()

print("Best EfficientNetV2-S checkpoint loaded!")

Best EfficientNetV2-S checkpoint loaded!


In [71]:
from tqdm.auto import tqdm

In [72]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

all_labels_eff = []
all_preds_eff = []

efficientnet.eval()

with torch.no_grad():
    for images, labels in tqdm(test_loader_eff, desc="EfficientNet Test Evaluation"):

        images = images.to(device)

        outputs = efficientnet(images)
        predictions = outputs.argmax(dim=1).cpu()

        all_preds_eff.extend(predictions.numpy())
        all_labels_eff.extend(labels.numpy())

print("Test predictions completed!")

EfficientNet Test Evaluation:   0%|          | 0/509 [00:00<?, ?it/s]

Test predictions completed!


In [73]:
test_accuracy_eff = accuracy_score(
    all_labels_eff, all_preds_eff
)

precision_eff, recall_eff, f1_eff, _ = (
    precision_recall_fscore_support(
        all_labels_eff,
        all_preds_eff,
        average="macro",
        zero_division=0
    )
)

print("EfficientNetV2-S Test Results")
print("-----------------------------")
print(f"Accuracy:  {test_accuracy_eff*100:.2f}%")
print(f"Precision: {precision_eff*100:.2f}%")
print(f"Recall:    {recall_eff*100:.2f}%")
print(f"Macro F1:  {f1_eff*100:.2f}%")

EfficientNetV2-S Test Results
-----------------------------
Accuracy:  98.46%
Precision: 97.92%
Recall:    97.70%
Macro F1:  97.78%


In [74]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

RESULT_DIR = Path("/content/drive/MyDrive/PlantLeafNet/results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Per-class classification report
report_eff = classification_report(
    all_labels_eff,
    all_preds_eff,
    target_names=classes,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report_eff).transpose().to_csv(
    RESULT_DIR / "efficientnet_test_classification_report.csv"
)

# Confusion matrix
cm_eff = confusion_matrix(all_labels_eff, all_preds_eff)

pd.DataFrame(
    cm_eff,
    index=classes,
    columns=classes
).to_csv(RESULT_DIR / "efficientnet_test_confusion_matrix.csv")

print("EfficientNet test reports saved to Google Drive!")

EfficientNet test reports saved to Google Drive!
